In [ ]:
import kagglehub

path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")

print("Path to dataset files:",path)

100%|██████████| 1.99M/1.99M [00:00<00:00, 2.30MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/jp797498e/twitter-entity-sentiment-analysis/versions/2


In [ ]:
import kagglehub

path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")

print("Path to dataset files:",path)

data_file_path = os.path.join(path, 'twitter_training.csv')

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(data_file_path)

# Display the first 5 rows to verify
display(df.head())

output_csv_path = 'processed_twitter_sentiment.csv'
df.to_csv(output_csv_path, index=False) # index=False prevents pandas from writing the DataFrame index as a column

print(f"Dataset successfully saved to {output_csv_path}")

Using Colab cache for faster access to the 'twitter-entity-sentiment-analysis' dataset.
Path to dataset files: /kaggle/input/twitter-entity-sentiment-analysis


,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


Dataset successfully saved to processed_twitter_sentiment.csv


To train a model for sentiment classification, we first need to prepare the data. This involves:
1.  **Renaming columns**: The dataset was loaded without proper headers, so we need to assign meaningful names.
2.  **Text Preprocessing**: Cleaning the tweet content by removing noise (URLs, mentions, special characters), converting to lowercase, removing stopwords, and lemmatizing words.
3.  **Sentiment Encoding**: Converting categorical sentiment labels ('Positive', 'Negative', 'Neutral', 'Irrelevant') into numerical format.
4.  **Data Splitting**: Dividing the dataset into training and testing sets.
5.  **Text Vectorization**: Converting the processed text into numerical features using TF-IDF.
6.  **Model Training**: Training a Logistic Regression model on the vectorized data.
7.  **Prediction Function**: Creating a reusable function to predict sentiment for new text inputs.

In [ ]:
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Ensure NLTK resources are downloaded (run once)
nltk.download('stopwords')
nltk.download('wordnet')

# 1. Correct column names
temp_column_names = ['ID', 'Entity', 'Sentiment', 'Tweet content', 'col_extra_1', 'col_extra_2']
df.columns = temp_column_names

# Now, we select only the 4 relevant columns and effectively drop the extra ones.
df = df[['ID', 'Entity', 'Sentiment', 'Tweet content']]

# Drop rows with missing 'Tweet content' or 'Sentiment'
df.dropna(subset=['Tweet content', 'Sentiment'], inplace=True)

# **Filter for only 'Positive' and 'Negative' sentiments**
df = df[df['Sentiment'].isin(['Positive', 'Negative'])]

# 2. Preprocess text data
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = str(text) # Ensure text is string
    text = text.lower() # Convert to lowercase
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Remove URLs
    text = re.sub(r'@[\w]*', '', text) # Remove mentions
    text = re.sub(r'#', '', text) # Remove hashtag symbol
    text = re.sub(r'[^a-z\s]', '', text) # Remove non-alphabetic characters
    tokens = text.split() # Tokenize
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words] # Lemmatize and remove stopwords
    return ' '.join(tokens)

df['processed_content'] = df['Tweet content'].apply(preprocess_text)

# 3. Encode sentiment labels
# Now, we will only have 'Positive' and 'Negative' in our sentiment labels.

sentiment_labels = df['Sentiment'].unique()
sentiment_mapping = {label: idx for idx, label in enumerate(sentiment_labels)}
df['sentiment_encoded'] = df['Sentiment'].map(sentiment_mapping)

reverse_sentiment_mapping = {idx: label for label, idx in sentiment_mapping.items()}

# 4. Split data
X = df['processed_content']
y = df['sentiment_encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 5. Vectorize text data (TF-IDF)
vectorizer = TfidfVectorizer(max_features=5000) # Limit features to prevent overly sparse data
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 6. Train the model (Logistic Regression)
model = LogisticRegression(max_iter=1000) # Increase max_iter for convergence, especially with more data
model.fit(X_train_tfidf, y_train)

# Evaluate the model
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy on Test Set (Positive/Negative only): {accuracy:.4f}")

# 7. Prediction function
def predict_sentiment(prompt):
    processed_prompt = preprocess_text(prompt)
    # Transform the new prompt using the same vectorizer fitted on training data
    prompt_tfidf = vectorizer.transform([processed_prompt])
    # Make prediction
    prediction_encoded = model.predict(prompt_tfidf)[0]
    # Convert encoded prediction back to original sentiment label
    predicted_sentiment = reverse_sentiment_mapping[prediction_encoded]
    return predicted_sentiment

print("\nModel training complete! You can now use the `predict_sentiment` function.")

# Example usage:
print("\n--- Example Predictions (Positive/Negative only) ---")
example_prompt_positive = "This game is absolutely amazing, I love playing it all day long!"
example_prompt_negative = "Worst customer service ever, I am so disappointed and angry."
example_prompt_neutral = "The meeting is scheduled for tomorrow at 10 AM." # This will now be classified as either Positive or Negative
example_prompt_irrelevant = "I just had breakfast, a cup of coffee and toast." # This will also be classified as either Positive or Negative

print(f"Prompt: '{example_prompt_positive}'\nSentiment: {predict_sentiment(example_prompt_positive)}\n")
print(f"Prompt: '{example_prompt_negative}'\nSentiment: {predict_sentiment(example_prompt_negative)}\n")
print(f"Prompt: '{example_prompt_neutral}'\nSentiment: {predict_sentiment(example_prompt_neutral)}\n")
print(f"Prompt: '{example_prompt_irrelevant}'\nSentiment: {predict_sentiment(example_prompt_irrelevant)}\n")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!



Model Accuracy on Test Set (Positive/Negative only): 0.8632

Model training complete! You can now use the `predict_sentiment` function.

--- Example Predictions (Positive/Negative only) ---
Prompt: 'This game is absolutely amazing, I love playing it all day long!'
Sentiment: Positive

Prompt: 'Worst customer service ever, I am so disappointed and angry.'
Sentiment: Negative

Prompt: 'The meeting is scheduled for tomorrow at 10 AM.'
Sentiment: Positive

Prompt: 'I just had breakfast, a cup of coffee and toast.'
Sentiment: Negative

